# Notebook 04 — Fine-Tuning AH-PC for Emotion Classification

Notebooks 02 and 03 built and pre-trained the AH-PC encoder using **self-supervised**
predictive coding — the model never saw an emotion label. It only learned to predict
future EEG states from a current one.

This notebook fine-tunes that pre-trained model for emotion classification, following
Sheibani's **subject-dependent 3-fold cross-validation** protocol exactly, so the
results are directly comparable to her numbers:

- **Subject-dependent 3-fold CV, split by trial.** Each subject contributes 45 trials
  (3 sessions × 15 trials). Trials 0–14 are session 1, 15–29 are session 2, 30–44 are
  session 3 (confirmed against the SEED-V label pattern in the pair files). Each fold
  holds out one full session's trials as the test set and trains on the other two —
  this is a leave-one-session-out split, done independently for each of the 16 subjects.
- **Partial freezing, not full freezing.** Sheibani's baseline freezes only the first 8
  layers of her CNN encoder and fine-tunes the rest. In AH-PC's architecture, the
  analogous "early, generic" component is the **BandAttention + ChannelAttention**
  gating (the very first thing the raw EEG segment passes through) — so we freeze
  those two modules only, and keep the CNN layers, bottleneck, and projector trainable.
- **A separate model is trained per subject per fold** (16 subjects × 3 folds = 48
  training runs), each with its own class-weighted loss and its own best checkpoint —
  matching Sheibani's per-subject-per-fold protocol rather than one pooled model.

## What this notebook does

1. Re-defines `AHPCModel` (identical to Notebooks 02/03) so the checkpoint can be loaded
2. Defines `AHPCClassifier`, which loads the pre-trained checkpoint, restores encoder +
   projector weights, and freezes **only** `BandAttention` + `ChannelAttention`
3. Adds a small classification head: `Linear(256→128) → ReLU → Dropout(0.3) → Linear(128→5)`
4. Builds the trial-based 3-fold split for any (subject, fold) pair from the pair `.pkl`
   files in `src/data/processed/`
5. Trains one `AHPCClassifier` per subject per fold with class-weighted loss
6. Reports per-subject mean accuracy, overall mean ± std across all 16 subjects, an
   aggregated classification report, and a normalized confusion matrix
7. Compares the overall mean accuracy against Sheibani's PC-SSL baseline (83.02% ± 13.12%)

## Smoke test

Full training (16 subjects × 3 folds × 25 epochs = 48 runs) takes a while, so it's built
but not launched automatically (`RUN_FULL_TRAINING = False`). This notebook *does*
execute a 2-epoch smoke test on Subject 1, Fold 1 to confirm the trial-based split is
correct and that `BandAttention`/`ChannelAttention` truly don't change while everything
else does.

In [1]:
import os
import copy
import pickle

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

DATA_DIR         = "/workspaces/AH-PC/src/data/processed"
PRETRAINED_CKPT  = "/workspaces/AH-PC/src/models/ahpc_pretrained_best.pt"
FINETUNE_CKPT_DIR = "/workspaces/AH-PC/src/models/finetune"

EMOTIONS = ["Disgust", "Fear", "Sad", "Neutral", "Happy"]
NUM_SUBJECTS = 16

PyTorch version: 2.12.1+cu130
Device: cpu


## 1. Re-defining the AH-PC Architecture

The pre-trained checkpoint stores raw tensor weights keyed by module name (a `state_dict`) —
it does not carry the class definitions with it. To load those weights back into a live model
we need the exact same class definitions used during pre-training, so every layer name and
shape lines up. These classes are copied verbatim from Notebooks 02/03 — nothing here is changed.

In [2]:
class BandAttention(nn.Module):
    def __init__(self, num_bands):
        super().__init__()
        self.fc1 = nn.Linear(num_bands * 2, num_bands // 2)
        self.fc2 = nn.Linear(num_bands // 2, num_bands)

    def forward(self, x):  # x: (B, C, B_bands)
        avg_pool = x.mean(dim=1)
        max_pool = x.max(dim=1).values
        context = torch.cat([avg_pool, max_pool], dim=-1)
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(1)


class ChannelAttention(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.fc1 = nn.Linear(num_channels * 2, num_channels // 2)
        self.fc2 = nn.Linear(num_channels // 2, num_channels)

    def forward(self, x):  # x: (B, C, B_bands)
        avg_pool = x.mean(dim=2)
        max_pool = x.max(dim=2).values
        context = torch.cat([avg_pool, max_pool], dim=-1)
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(context))))
        return x * weights.unsqueeze(2)


class SheibaniEncoder(nn.Module):
    '''Sheibani's attention + CNN encoder, reproduced exactly (Notebook 02).'''

    OUT_CHANNELS = 64

    def __init__(self):
        super().__init__()
        self.band_attn = BandAttention(num_bands=5)
        self.channel_attn = ChannelAttention(num_channels=62)

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

    def forward(self, x):  # x: (B, 62, 5)
        x = self.band_attn(x)
        x = self.channel_attn(x)
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.bottleneck(x)
        return x  # (B, 64, 15, 5)


class HorizonAttentionModule(nn.Module):
    def __init__(self, latent_dim=256, num_horizons=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, num_horizons),
        )

    def forward(self, z):
        return F.softmax(self.net(z), dim=-1)


class HorizonDecoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, latent_dim),
        )

    def forward(self, z):
        return self.net(z)


HORIZONS = [1, 2, 4, 8]
LATENT_DIM = 256


class AHPCModel(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM, horizons: list = HORIZONS):
        super().__init__()
        self.horizons = horizons

        self.sheibani_encoder = SheibaniEncoder()

        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 2)),
            nn.Flatten(),
            nn.Linear(SheibaniEncoder.OUT_CHANNELS * 4 * 2, latent_dim),
            nn.LayerNorm(latent_dim),
        )

        self.horizon_attn = HorizonAttentionModule(
            latent_dim=latent_dim, num_horizons=len(horizons),
        )
        self.decoders = nn.ModuleList([
            HorizonDecoder(latent_dim=latent_dim) for _ in horizons
        ])

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.sheibani_encoder(x)
        return self.projector(feat)

    def forward(self, anchor: torch.Tensor):
        z_t = self.get_embedding(anchor)
        alpha = self.horizon_attn(z_t)
        predicted_embeddings = {
            k: self.decoders[i](z_t) for i, k in enumerate(self.horizons)
        }
        return z_t, alpha, predicted_embeddings


print("AHPCModel class re-defined (matches Notebooks 02/03).")

AHPCModel class re-defined (matches Notebooks 02/03).


## 2. Loading the Pre-Trained Encoder — Partial Freezing

Only the encoder + projector are carried over from `AHPCModel` (the horizon attention
module and the four decoder heads only existed to compute the pre-training loss, and
play no role in classification).

Unlike a full linear-probe setup, **we don't freeze the whole encoder**. Sheibani's
baseline freezes only the first 8 layers of her CNN and fine-tunes the rest — the idea
being that early layers learn generic, low-level filters that transfer well as-is, while
later layers need to adapt to the new task. Translated into AH-PC's architecture, the
earliest, most generic component is the **BandAttention + ChannelAttention** gating
(`SheibaniEncoder.band_attn` / `.channel_attn`) — the very first thing a raw EEG segment
passes through, before any convolution happens. So:

- **Frozen:** `band_attn`, `channel_attn` (`requires_grad = False`, permanently `.eval()`)
- **Trainable:** the CNN (`encoder`), the `bottleneck`, the `projector`, and the new head

`AHPCClassifier` below:
1. Instantiates a fresh `SheibaniEncoder` + projector with the exact same shapes as `AHPCModel`
2. Loads `ahpc_pretrained_best.pt` and restores the `sheibani_encoder.*` / `projector.*`
   weight tensors from the full checkpoint's `state_dict` — this gives every layer,
   frozen or not, a pre-trained starting point rather than random init
3. Sets `requires_grad = False` only on `band_attn`/`channel_attn` parameters
4. Overrides `.train()` so `band_attn`/`channel_attn` are always forced into `eval()`
   mode (no dropout/BN drift there — though note neither module actually contains
   BatchNorm, so this mainly guards against future changes), while the rest of the
   model follows the normal train/eval mode of its parent
5. Adds the classification head: `Linear(256→128) → ReLU → Dropout(0.3) → Linear(128→5)`

In [3]:
class AHPCClassifier(nn.Module):
    '''
    AH-PC encoder + projector with partial freezing (matches Sheibani's baseline):
    BandAttention/ChannelAttention are frozen, everything else (CNN, bottleneck,
    projector, head) is trainable.
    '''

    def __init__(self, pretrained_ckpt_path, num_classes=5, latent_dim=LATENT_DIM):
        super().__init__()

        self.encoder = SheibaniEncoder()
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 2)),
            nn.Flatten(),
            nn.Linear(SheibaniEncoder.OUT_CHANNELS * 4 * 2, latent_dim),
            nn.LayerNorm(latent_dim),
        )
        self._load_pretrained(pretrained_ckpt_path)

        # Freeze only the attention gating modules — the CNN, bottleneck, and
        # projector all remain trainable, matching Sheibani's "freeze first 8 layers,
        # fine-tune the rest" baseline protocol.
        for p in self.encoder.band_attn.parameters():
            p.requires_grad = False
        for p in self.encoder.channel_attn.parameters():
            p.requires_grad = False

        self.head = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def _load_pretrained(self, path):
        # weights_only=False: this is our own checkpoint (Notebook 03), which stores
        # plain Python/numpy history alongside the tensors, not an untrusted file.
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        full_sd = ckpt["model_state_dict"]

        enc_sd = {
            k[len("sheibani_encoder."):]: v
            for k, v in full_sd.items() if k.startswith("sheibani_encoder.")
        }
        proj_sd = {
            k[len("projector."):]: v
            for k, v in full_sd.items() if k.startswith("projector.")
        }
        self.encoder.load_state_dict(enc_sd)
        self.projector.load_state_dict(proj_sd)
        print(f"Restored {len(enc_sd)} encoder tensors and {len(proj_sd)} projector "
              f"tensors from {path}")

    def train(self, mode: bool = True):
        # BandAttention/ChannelAttention stay in eval mode regardless of the outer
        # model's mode. (Neither module has BatchNorm/Dropout, so this is mostly a
        # safeguard for architecture changes, not currently load-bearing.)
        super().train(mode)
        self.encoder.band_attn.eval()
        self.encoder.channel_attn.eval()
        return self

    def get_embedding(self, x):
        feat = self.encoder(x)
        return self.projector(feat)

    def forward(self, x):
        z = self.get_embedding(x)
        return self.head(z)


_model = AHPCClassifier(PRETRAINED_CKPT)
_frozen_params = list(_model.encoder.band_attn.parameters()) + list(_model.encoder.channel_attn.parameters())
_trainable_params = [p for p in _model.parameters() if p.requires_grad]
n_frozen = sum(p.numel() for p in _frozen_params)
n_trainable = sum(p.numel() for p in _trainable_params)
print(f"\nFrozen params (BandAttention + ChannelAttention): {n_frozen:,}")
print(f"Trainable params (CNN + bottleneck + projector + head): {n_trainable:,}")
del _model, _frozen_params, _trainable_params

Restored 29 encoder tensors and 4 projector tensors from /workspaces/AH-PC/src/models/ahpc_pretrained_best.pt

Frozen params (BandAttention + ChannelAttention): 5,896
Trainable params (CNN + bottleneck + projector + head): 188,901


## 3. Dataset: Subject-Dependent 3-Fold CV, Split by Trial

Sheibani's protocol evaluates **within each subject**, not on a pooled random split.
Each subject's 45 trials are 3 sessions of 15 trials each; trials are laid out
contiguously in the pair files as `[session 1: 0-14][session 2: 15-29][session 3: 30-44]`
(verified against the SEED-V per-session label pattern). Each fold holds out one
session's trials as the test set and trains on the remaining two sessions —
a leave-one-session-out split, computed independently per subject:

- **Fold 1:** train = sessions 2+3 (trials 15-44), test = session 1 (trials 0-14)
- **Fold 2:** train = sessions 1+3 (trials 0-14, 30-44), test = session 2 (trials 15-29)
- **Fold 3:** train = sessions 1+2 (trials 0-29), test = session 3 (trials 30-44)

The split is done purely by each pair's `'trial'` field — every pair generated from a
trial in the test session goes to test, everything else goes to train. This keeps all
segments from a given trial on the same side of the split, unlike the pooled
label-stratified split used previously, which could leak near-duplicate segments from
the same trial across train/test.

In [4]:
class EmotionDataset(Dataset):
    def __init__(self, pairs):
        self.samples = pairs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p = self.samples[idx]
        anchor = torch.tensor(p["anchor"], dtype=torch.float32)  # (62, 5)
        label = torch.tensor(p["label"], dtype=torch.long)
        return anchor, label


SESSION_TRIALS = [list(range(0, 15)), list(range(15, 30)), list(range(30, 45))]


def get_fold_trials(fold_idx: int):
    """fold_idx in {0, 1, 2}. Returns (train_trials, test_trials) for that fold,
    holding out one session's trials as test and training on the other two."""
    test_trials = SESSION_TRIALS[fold_idx]
    train_trials = [t for i, s in enumerate(SESSION_TRIALS) if i != fold_idx for t in s]
    return sorted(train_trials), sorted(test_trials)


def load_subject_pairs(subject_id: int, data_dir=DATA_DIR):
    path = os.path.join(data_dir, f"subject_{subject_id:02d}_pairs.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_subject_fold_pairs(subject_id: int, fold_idx: int):
    """Split one subject's pairs into train/test by trial index for the given fold."""
    pairs = load_subject_pairs(subject_id)
    train_trials, test_trials = get_fold_trials(fold_idx)
    train_set, test_set = set(train_trials), set(test_trials)

    train_pairs = [p for p in pairs if p["trial"] in train_set]
    test_pairs = [p for p in pairs if p["trial"] in test_set]
    return train_pairs, test_pairs, train_trials, test_trials


print("Trial-based fold splitting defined.")
for fold_idx in range(3):
    train_trials, test_trials = get_fold_trials(fold_idx)
    print(f"  Fold {fold_idx + 1}: train trials={train_trials[0]}-{train_trials[-1]} "
          f"(n={len(train_trials)}), test trials={test_trials[0]}-{test_trials[-1]} "
          f"(n={len(test_trials)})")

Trial-based fold splitting defined.
  Fold 1: train trials=15-44 (n=30), test trials=0-14 (n=15)
  Fold 2: train trials=0-44 (n=30), test trials=15-29 (n=15)
  Fold 3: train trials=0-29 (n=30), test trials=30-44 (n=15)


## 4. Class Imbalance and Inverse-Frequency Weights

Because training is now per-subject-per-fold, each fold's train split has its own class
distribution (a 15-trial session can easily be skewed toward one emotion). Instead of a
single global weight vector, we compute **inverse-frequency weights from that fold's own
training labels** and pass them into a weighted `CrossEntropyLoss` for that run.

In [5]:
def compute_class_weights(labels, num_classes=5):
    """Inverse-frequency weights from a fold's own training labels.
    Classes absent from a small per-subject fold are clipped to count=1 to avoid div-by-zero."""
    counts = np.bincount(labels, minlength=num_classes)
    counts = np.clip(counts, 1, None)
    weights = counts.max() / counts
    return torch.tensor(weights, dtype=torch.float32)


# Example: Subject 1, Fold 1 training labels
_train_pairs, _test_pairs, _, _ = get_subject_fold_pairs(1, 0)
_train_labels = np.array([p["label"] for p in _train_pairs])
_weights = compute_class_weights(_train_labels)

print("Example — Subject 1, Fold 1 training-fold class weights:")
for name, count, weight in zip(EMOTIONS, np.bincount(_train_labels, minlength=5), _weights.tolist()):
    print(f"  {name:10s}  count={int(count):5d}  weight={weight:.3f}")
del _train_pairs, _test_pairs, _train_labels, _weights

Example — Subject 1, Fold 1 training-fold class weights:
  Disgust     count=  149  weight=1.718
  Fear        count=  184  weight=1.391
  Sad         count=  256  weight=1.000
  Neutral     count=  185  weight=1.384
  Happy       count=  128  weight=2.000


## 5. Training: One Model Per Subject Per Fold

`train_subject_fold(subject_id, fold_idx)` builds the trial-based train/test split for
that one (subject, fold) pair, trains a fresh `AHPCClassifier` (re-loaded from the
pre-trained checkpoint each time, so folds never share weights), and evaluates on the
held-out session at the end of every epoch. Hyperparameters match Sheibani's protocol:

- **Optimizer:** AdamW, `lr=8e-5`, `weight_decay=1e-4`
- **Scheduler:** `StepLR(step_size=10, gamma=0.5)`
- **Loss:** `CrossEntropyLoss` with that fold's inverse-frequency class weights
- **Epochs:** 25 · **Batch size:** 32

The best-test-accuracy epoch's weights are saved to
`src/models/finetune/subject{sid:02d}_fold{fold}_best.pt`, and its predictions are kept
for the aggregated report in Section 7.

In [6]:
def train_subject_fold(subject_id, fold_idx, epochs=25, batch_size=32, lr=8e-5,
                        weight_decay=1e-4, step_size=10, gamma=0.5,
                        device=DEVICE, verbose=True):
    train_pairs, test_pairs, train_trials, test_trials = get_subject_fold_pairs(subject_id, fold_idx)

    train_ds = EmotionDataset(train_pairs)
    test_ds = EmotionDataset(test_pairs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    train_labels = np.array([p["label"] for p in train_pairs])
    class_weights = compute_class_weights(train_labels).to(device)

    model = AHPCClassifier(PRETRAINED_CKPT).to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_acc = -1.0
    best_state = None
    best_preds, best_labels = None, None
    history = {"train_loss": [], "test_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for anchor, label in train_loader:
            anchor, label = anchor.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(anchor)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * anchor.size(0)
        scheduler.step()
        avg_loss = running_loss / len(train_ds)

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for anchor, label in test_loader:
                logits = model(anchor.to(device))
                preds.extend(logits.argmax(dim=1).cpu().numpy())
                labels.extend(label.numpy())
        acc = accuracy_score(labels, preds)

        history["train_loss"].append(avg_loss)
        history["test_acc"].append(acc)

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())
            best_preds, best_labels = preds, labels

        if verbose:
            print(f"    Epoch {epoch:2d}/{epochs}  loss={avg_loss:.4f}  test_acc={acc:.4f}")

    os.makedirs(FINETUNE_CKPT_DIR, exist_ok=True)
    ckpt_path = os.path.join(FINETUNE_CKPT_DIR, f"subject{subject_id:02d}_fold{fold_idx + 1}_best.pt")
    torch.save({"model_state_dict": best_state, "best_acc": best_acc}, ckpt_path)

    return {
        "subject": subject_id,
        "fold": fold_idx + 1,
        "accuracy": best_acc,
        "y_true": np.array(best_labels),
        "y_pred": np.array(best_preds),
        "history": history,
        "ckpt_path": ckpt_path,
        "n_train": len(train_ds),
        "n_test": len(test_ds),
    }


print("train_subject_fold() defined.")

train_subject_fold() defined.


## 6. Full Experiment: All 16 Subjects × 3 Folds

Runs `train_subject_fold` for every subject/fold combination (48 training runs of 25
epochs each). **Intentionally left off** (`RUN_FULL_TRAINING = False`) until the pipeline
has been validated by the smoke test in Section 9 — flip the flag to `True` to launch
the full protocol. Results are accumulated in `fold_results`, one dict per (subject, fold).

In [7]:
RUN_FULL_TRAINING = False  # set True to launch all 16 subjects x 3 folds (48 runs x 25 epochs)

fold_results = []  # one dict per (subject, fold), from train_subject_fold()

if RUN_FULL_TRAINING:
    for subject_id in range(1, NUM_SUBJECTS + 1):
        print(f"\nSubject {subject_id}")
        for fold_idx in range(3):
            print(f"  Fold {fold_idx + 1}/3")
            res = train_subject_fold(subject_id, fold_idx, epochs=25, batch_size=32, verbose=False)
            fold_results.append(res)
            print(f"    -> best test_acc = {res['accuracy']:.4f}")
    print("\nTraining complete for all subjects and folds.")
else:
    print("RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. "
          "See Section 9 for a smoke test of the pipeline instead.")

RUN_FULL_TRAINING is False — skipping the full 16-subject x 3-fold run. See Section 9 for a smoke test of the pipeline instead.


## 7. Evaluation

Once `fold_results` is populated (after a full run), we report, in the same format as
Sheibani's original protocol:

- **Per-subject mean accuracy** across its 3 folds (± std)
- **Overall mean ± std across all 16 subjects** (computed over the 16 per-subject means)
- An **aggregated classification report** (precision/recall/F1 per emotion) pooling every
  subject's best-fold predictions together
- A **normalized confusion matrix** over the same pooled predictions

In [8]:
from collections import defaultdict


def plot_confusion_matrix(cm, class_names=EMOTIONS, title="Confusion Matrix (Normalized)"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, label="Fraction of true class")
    plt.tight_layout()
    plt.show()


overall_mean = overall_std = None

if not fold_results:
    print("No results yet — set RUN_FULL_TRAINING=True in Section 6 and re-run to "
          "produce evaluation metrics.")
else:
    acc_by_subject = defaultdict(list)
    for r in fold_results:
        acc_by_subject[r["subject"]].append(r["accuracy"])

    print("Per-subject mean accuracy across 3 folds:")
    subject_means = []
    for sid in sorted(acc_by_subject):
        accs = acc_by_subject[sid]
        m, s = np.mean(accs), np.std(accs)
        subject_means.append(m)
        fold_str = ", ".join(f"{a * 100:.2f}%" for a in accs)
        print(f"  Subject {sid:2d}: {m * 100:5.2f}% ± {s * 100:5.2f}%  (folds: {fold_str})")

    overall_mean = float(np.mean(subject_means) * 100)
    overall_std = float(np.std(subject_means) * 100)
    print(f"\nOverall mean accuracy across all {len(subject_means)} subjects: "
          f"{overall_mean:.2f}% ± {overall_std:.2f}%")

    all_y_true = np.concatenate([r["y_true"] for r in fold_results])
    all_y_pred = np.concatenate([r["y_pred"] for r in fold_results])

    print("\nAggregated classification report (all subjects, all folds):")
    print(classification_report(all_y_true, all_y_pred, target_names=EMOTIONS, digits=3))

    agg_cm = confusion_matrix(all_y_true, all_y_pred, labels=[0, 1, 2, 3, 4])
    plot_confusion_matrix(agg_cm, title="Aggregated Confusion Matrix (Normalized) — All Subjects/Folds")

No results yet — set RUN_FULL_TRAINING=True in Section 6 and re-run to produce evaluation metrics.


## 8. Comparison Against the PC-SSL Baseline

Sheibani's PC-SSL baseline reports **83.02% ± 13.12%** on this exact subject-dependent
3-fold CV protocol. Because the fold definitions, freezing scheme, and hyperparameters
here match hers, this is a direct apples-to-apples comparison.

In [9]:
PC_SSL_BASELINE_MEAN = 83.02
PC_SSL_BASELINE_STD = 13.12

if overall_mean is None:
    print("Run the full 16-subject x 3-fold protocol (Section 6) to compare against "
          "the PC-SSL baseline.")
else:
    diff = overall_mean - PC_SSL_BASELINE_MEAN
    print(f"{'PC-SSL baseline (Sheibani):':28s} {PC_SSL_BASELINE_MEAN:.2f}% ± {PC_SSL_BASELINE_STD:.2f}%")
    print(f"{'AH-PC (this run):':28s} {overall_mean:.2f}% ± {overall_std:.2f}%")
    print(f"{'Difference:':28s} {diff:+.2f} pts")

Run the full 16-subject x 3-fold protocol (Section 6) to compare against the PC-SSL baseline.


## 9. Smoke Test

Before launching the full 48-run protocol, we validate the pipeline end-to-end with 2
epochs on **Subject 1, Fold 1** — the real trial-based split built in Section 3. This
checks:

1. **The trial-based split is correct.** We print the train/test pair counts and the
   exact trial indices on each side, so it's plain to see session 1's trials (0-14) are
   held out and sessions 2+3 (15-44) are used for training.
2. **`BandAttention`/`ChannelAttention` truly don't change.** We snapshot every weight
   tensor in those two modules before training and compare it byte-for-byte to the same
   tensors after training.
3. **Everything else does change.** The CNN encoder, bottleneck, projector, and
   classification head should all have different weights after training.
4. **Predictions come out in a sane shape.** A handful of predicted vs. actual labels,
   just to eyeball that nothing is index-shifted or the wrong dtype.

This is not a claim about accuracy — 2 epochs tells us nothing about whether AH-PC beats
the baseline, only that the code is correct.

In [10]:
print("=" * 70)
print("SMOKE TEST: 2 epochs on Subject 1, Fold 1 only")
print("=" * 70)

SMOKE_SUBJECT = 1
SMOKE_FOLD = 0  # Fold 1 (0-based)
SMOKE_EPOCHS = 2

train_pairs, test_pairs, train_trials, test_trials = get_subject_fold_pairs(SMOKE_SUBJECT, SMOKE_FOLD)

print(f"Train trials ({len(train_trials)}): {train_trials}")
print(f"Test trials  ({len(test_trials)}): {test_trials}")
print(f"Train pairs: {len(train_pairs):,}")
print(f"Test pairs:  {len(test_pairs):,}\n")

train_ds = EmotionDataset(train_pairs)
test_ds = EmotionDataset(test_pairs)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

train_labels = np.array([p["label"] for p in train_pairs])
class_weights = compute_class_weights(train_labels).to(DEVICE)

model = AHPCClassifier(PRETRAINED_CKPT).to(DEVICE)

frozen_before = {
    "band_attn": {k: v.clone() for k, v in model.encoder.band_attn.state_dict().items()},
    "channel_attn": {k: v.clone() for k, v in model.encoder.channel_attn.state_dict().items()},
}
trainable_before = {
    "cnn": {k: v.clone() for k, v in model.encoder.encoder.state_dict().items()},
    "bottleneck": {k: v.clone() for k, v in model.encoder.bottleneck.state_dict().items()},
    "projector": {k: v.clone() for k, v in model.projector.state_dict().items()},
    "head": {k: v.clone() for k, v in model.head.state_dict().items()},
}

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=8e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(1, SMOKE_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for anchor, label in train_loader:
        anchor, label = anchor.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(anchor)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * anchor.size(0)
    scheduler.step()
    print(f"  Epoch {epoch}/{SMOKE_EPOCHS}  train_loss={running_loss / len(train_ds):.4f}")

frozen_after = {
    "band_attn": {k: v.clone() for k, v in model.encoder.band_attn.state_dict().items()},
    "channel_attn": {k: v.clone() for k, v in model.encoder.channel_attn.state_dict().items()},
}
trainable_after = {
    "cnn": {k: v.clone() for k, v in model.encoder.encoder.state_dict().items()},
    "bottleneck": {k: v.clone() for k, v in model.encoder.bottleneck.state_dict().items()},
    "projector": {k: v.clone() for k, v in model.projector.state_dict().items()},
    "head": {k: v.clone() for k, v in model.head.state_dict().items()},
}

print()
all_frozen_unchanged = True
for name, before in frozen_before.items():
    after = frozen_after[name]
    unchanged = all(torch.equal(before[k], after[k]) for k in before)
    all_frozen_unchanged &= unchanged
    print(f"{name:14s} unchanged after training: {unchanged}")

print()
all_trainable_changed = True
for name, before in trainable_before.items():
    after = trainable_after[name]
    changed = any(not torch.equal(before[k], after[k]) for k in before)
    all_trainable_changed &= changed
    print(f"{name:14s} changed after training:   {changed}")

assert all_frozen_unchanged, "FAIL: a frozen module (band_attn/channel_attn) changed during training!"
assert all_trainable_changed, "FAIL: a module that should be trainable did not change — check requires_grad/optimizer wiring!"
print("\nAll smoke-test assertions passed.")


SMOKE TEST: 2 epochs on Subject 1, Fold 1 only
Train trials (30): [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Test trials  (15): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Train pairs: 902
Test pairs:  561

Restored 29 encoder tensors and 4 projector tensors from /workspaces/AH-PC/src/models/ahpc_pretrained_best.pt


  Epoch 1/2  train_loss=1.6239


  Epoch 2/2  train_loss=1.6117

band_attn      unchanged after training: True
channel_attn   unchanged after training: True

cnn            changed after training:   True
bottleneck     changed after training:   True
projector      changed after training:   True
head           changed after training:   True

All smoke-test assertions passed.


In [11]:
model.eval()
anchor_sample, label_sample = next(iter(test_loader))
with torch.no_grad():
    logits = model(anchor_sample.to(DEVICE))
    probs = F.softmax(logits, dim=-1)
    preds = probs.argmax(dim=1).cpu()

print("Sample predictions vs actual labels (2 epochs only — expect near-chance):\n")
n_show = min(10, len(preds))
for i in range(n_show):
    pred_name = EMOTIONS[preds[i].item()]
    true_name = EMOTIONS[label_sample[i].item()]
    match = "✓" if pred_name == true_name else " "
    print(f"  [{match}] Predicted: {pred_name:10s}  Actual: {true_name:10s}")

smoke_acc = accuracy_score(label_sample.numpy(), preds.numpy())
print(f"\nSmoke-test batch accuracy: {smoke_acc:.3f} (n={len(preds)}, chance level = 0.20)")

Sample predictions vs actual labels (2 epochs only — expect near-chance):

  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     
  [ ] Predicted: Neutral     Actual: Happy     

Smoke-test batch accuracy: 0.188 (n=32, chance level = 0.20)
